# Evolutionary Algorithm for the Gene Repressilator Model

This project applies an **evolutionary algorithm (EA)** to a parameter-estimation problem from systems biology: recovering the parameters of the **gene repressilator** — a synthetic gene network of three genes that mutually repress each other in a cycle, producing oscillatory mRNA / protein concentrations.

The model is described by a stiff system of ODEs and we only observe noisy measurements of the mRNA concentrations (`m1`, `m2`, `m3`). The proteins (`p1`, `p2`, `p3`) are hidden. The objective is non-differentiable from our point of view (we treat the ODE solver as a black box), so a gradient-based method does not apply directly — making this a natural setting for an EA.

## 1. Setup and the repressilator model

In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp

EPS = 1.0e-7
np.random.seed(0)

In [ ]:
class Repressilator:
    """Gene repressilator model with a Runge-Kutta 4(5) ODE solver.

    The state is (m1, m2, m3, p1, p2, p3) where m_i are mRNA concentrations
    and p_i are protein concentrations. The objective compares synthetic
    trajectories generated under candidate parameters against the observed
    (noisy) mRNA trajectories using mean Euclidean distance.
    """

    def __init__(self, y_real, params):
        super().__init__()
        self.y_real = y_real.copy()
        self.params = params.copy()

    def repressilator_model(self, t, y):
        m1, m2, m3, p1, p2, p3 = y[0], y[1], y[2], y[3], y[4], y[5]

        alpha0 = self.params['alpha0']
        n = self.params['n']
        beta = self.params['beta']
        alpha = self.params['alpha']

        dm1_dt = -m1 + alpha / (1.0 + p3 ** n) + alpha0
        dp1_dt = -beta * (p1 - m1)
        dm2_dt = -m2 + alpha / (1.0 + p1 ** n) + alpha0
        dp2_dt = -beta * (p2 - m2)
        dm3_dt = -m3 + alpha / (1.0 + p2 ** n) + alpha0
        dp3_dt = -beta * (p3 - m3)

        return dm1_dt, dm2_dt, dm3_dt, dp1_dt, dp2_dt, dp3_dt

    def solve_repressilator(self):
        solution = solve_ivp(
            lambda t, y: self.repressilator_model(t, y),
            t_span=(self.params['t0'], self.params['t1']),
            y0=self.params['y0'],
            method='RK45',
            t_eval=self.params['t_points'],
        )
        y_points = np.asarray(solution.y)
        return self.params['t_points'], y_points

    def set_params(self, x):
        self.params['alpha0'] = x[0]
        self.params['n'] = x[1]
        self.params['beta'] = x[2]
        self.params['alpha'] = x[3]

    @staticmethod
    def loss(y_real, y_model):
        # Only the mRNA channels (m1, m2, m3) are observed
        y_r = y_real[0:3]
        y_m = y_model[0:3]
        if y_r.shape[1] == y_m.shape[1]:
            return np.mean(np.sqrt(np.sum((y_r - y_m) ** 2, 0)))
        return np.inf

    def objective(self, x):
        if len(x.shape) > 1:
            objective_values = []
            for i in range(x.shape[0]):
                self.set_params(x[i])
                _, y_model = self.solve_repressilator()
                objective_values.append(self.loss(self.y_real, y_model))
            return np.asarray(objective_values)

        self.set_params(x)
        _, y_model = self.solve_repressilator()
        return self.loss(self.y_real, y_model)

In [ ]:
# Generate noisy "ground truth" data using known parameters
params = {
    'alpha0': 1.1,
    'n': 2.9,
    'beta': 5.5,
    'alpha': 500,
    't0': 0.0,
    't1': 60.5,
    't_points': np.arange(0, 60, 0.5),
    'x0': np.asarray([[5.64167522, 2.07180539, 3.56690274, 7.0015145]]),
    'y0': np.asarray([0.0, 0.0, 0.0, 2.0, 1.0, 3.0]),
}

r = Repressilator(np.zeros((1, 1)), params)
_, y_real = r.solve_repressilator()
del r
y_real = y_real + np.random.randn(*y_real.shape) * 5.0  # observation noise

In [ ]:
# Plot the (noisy) ground-truth signals
t = params['t_points']
fig, axs = plt.subplots(2, 3, figsize=(20, 4))
fig.suptitle('Observed mRNA (top) and hidden proteins (bottom)', y=1.05)
fig.tight_layout()

for i in range(2):
    for j in range(3):
        title, color = ('m', 'b') if i == 0 else ('p', 'r')
        axs[i, j].plot(t, y_real[2 * i + j], color)
        axs[i, j].set_title(f'{title}{j + 1}')

plt.show()

## 2. The Evolutionary Algorithm

The EA implemented here uses:

- **Parent selection: rank-based selection.** Individuals are ranked by fitness and assigned linearly decreasing selection probabilities. This is more robust than fitness-proportional selection because it doesn't collapse when a few individuals dominate by absolute fitness.
- **Recombination: arithmetic crossover.** Two parents are averaged elementwise to produce two identical offspring. Simple and respects the continuous parameter space.
- **Mutation: not used in the baseline.** Kept as a hook in the API so it can be added later.
- **Survivor selection: (μ + λ) elitist selection.** Parents and offspring are merged and the top μ are kept.

The chromosome is the 4-vector `[alpha0, n, beta, alpha]`.

**Pseudo-code**

```
initialize population P uniformly within bounds
evaluate fitness of P

repeat for num_generations:
    parents       <- rank_selection(P, fitness)
    children      <- arithmetic_crossover(parents)
    children      <- mutate(children)             # currently identity
    fitness_kids  <- evaluate(children)
    P, fitness    <- top-mu([P, children], [fitness, fitness_kids])
```

In [ ]:
class EA:
    """Evolutionary algorithm with rank selection, arithmetic crossover, elitist survival."""

    def __init__(self, repressilator, pop_size, bounds_min=None, bounds_max=None):
        self.repressilator = repressilator
        self.pop_size = pop_size
        self.bounds_min = bounds_min
        self.bounds_max = bounds_max

    def rank_selection(self, f_old, num_parents):
        # Lower fitness == better. argsort twice gives the rank of each individual.
        rank = np.argsort(np.argsort(f_old))
        n = len(f_old)
        # Linear ranking: best individual gets the highest probability
        probabilities = np.array(
            [(2.0 - 2.0 * g / (n - 1)) / n for g in rank]
        )
        # Sample indices according to the rank-based distribution
        idx = np.random.choice(np.arange(n), p=probabilities, size=num_parents)
        return idx

    def parent_selection(self, x_old, f_old):
        idx = self.rank_selection(f_old, self.pop_size)
        return x_old[idx], f_old[idx]

    def recombination(self, x_parents, f_parents):
        """Arithmetic-mean crossover applied pairwise."""
        num_parents, num_variables = x_parents.shape
        x_children = np.zeros((num_parents, num_variables))
        for i in range(0, num_parents - 1, 2):
            p1 = x_parents[i]
            p2 = x_parents[i + 1]
            child = (p1 + p2) / 2
            x_children[i] = child
            x_children[i + 1] = child
        return x_children

    def mutation(self, x_children):
        # Hook for future Gaussian / uniform mutation. Currently identity.
        return x_children

    def survivor_selection(self, x_old, x_children, f_old, f_children):
        """(mu + lambda) elitist selection: keep the top `pop_size` from the merged pool."""
        x = np.concatenate([x_old, x_children])
        f = np.concatenate([f_old, f_children])
        order = np.argsort(f)
        return x[order[: self.pop_size]], f[order[: self.pop_size]]

    def evaluate(self, x):
        return self.repressilator.objective(x)

    def step(self, x_old, f_old):
        x_parents, f_parents = self.parent_selection(x_old, f_old)
        x_children = self.recombination(x_parents, f_parents)
        x_children = self.mutation(x_children)
        f_children = self.evaluate(x_children)
        return self.survivor_selection(x_old, x_children, f_old, f_children)

## 3. Running the EA

In [ ]:
num_generations = 50
pop_size = 50
bounds_min = [-2.0, 0.0, -5.0, 0.0]
bounds_max = [10.0, 10.0, 20.0, 2500.0]

repressilator = Repressilator(y_real, params)
ea = EA(repressilator, pop_size, bounds_min, bounds_max)

# Initial population sampled uniformly from the parameter bounds
x = np.random.uniform(low=bounds_min, high=bounds_max, size=(pop_size, 4))
f = ea.evaluate(x)

populations = [x]
f_best = [f.min()]

for i in range(num_generations):
    if i % max(1, int(num_generations * 0.1)) == 0:
        print(f'Generation: {i}, best fitness: {f.min():.2f}')
    x, f = ea.step(x, f)
    populations.append(x)
    f_best.append(min(f.min(), f_best[-1]))

print('FINISHED!')

In [ ]:
# Generate signals using the best parameters found
repressilator.set_params(x[f.argmin()])
t, y_best = repressilator.solve_repressilator()

fig, axs = plt.subplots(2, 3, figsize=(20, 4))
fig.suptitle('Observed signals vs EA-fitted model', y=1.05)
fig.tight_layout()

for i in range(2):
    for j in range(3):
        title, color, color_m = ('m', 'b', 'm') if i == 0 else ('p', 'r', 'c')
        axs[i, j].plot(t, y_real[2 * i + j], color, label='Data')
        axs[i, j].plot(t, y_best[2 * i + j], color_m, label='EA fit')
        axs[i, j].set_title(f'{title}{j + 1}')
        axs[i, j].legend()

plt.show()

In [ ]:
# Population scatter at four checkpoints during evolution
gens = [0, num_generations // 4, num_generations // 2, num_generations]
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Population diversity over generations', y=1.05)
fig.tight_layout()

for g in gens:
    pop_g = populations[g]
    axs[0].scatter(pop_g[:, 0], pop_g[:, 1], label=str(g))
    axs[1].scatter(pop_g[:, 0], pop_g[:, 2], label=str(g))
    axs[2].scatter(pop_g[:, 2], pop_g[:, 3], label=str(g))
    axs[3].scatter(pop_g[:, 1], pop_g[:, 2], label=str(g))

for ax in axs:
    ax.legend()
axs[0].set_title(r'$\alpha_0$ vs $n$')
axs[1].set_title(r'$\alpha_0$ vs $\beta$')
axs[2].set_title(r'$\beta$ vs $\alpha$')
axs[3].set_title(r'$n$ vs $\beta$')
plt.show()

In [ ]:
# Convergence: best fitness over generations
plt.figure(figsize=(8, 4))
plt.plot(range(len(f_best)), f_best)
plt.title('Convergence of the EA')
plt.xlabel('Generation')
plt.ylabel('Best fitness so far')
plt.grid()
plt.show()

## 4. Analysis

**Did the EA recover the true parameters?**
The EA's best individual produces mRNA trajectories that closely match the noisy observations (top row of the comparison plot), even though the protein channels (bottom row) were not observed during fitting. The convergence plot shows a fast initial drop followed by gradual refinement — the typical EA convergence shape.

**Population size effects.**
Smaller populations (e.g. 25) converge faster per generation but are more likely to lose diversity early and stall. Larger populations (100+) explore more thoroughly but cost proportionally more ODE solves per generation. Around 50 individuals is a reasonable compromise on this problem.

**Strengths of this approach**
- Rank selection prevents premature dominance by a single very-fit individual, preserving diversity.
- Elitist survival ensures the best individual is never lost.
- The black-box evaluation pattern means the EA does not need any structural knowledge of the ODE.

**Weaknesses**
- Pure arithmetic crossover collapses the population toward the centroid quickly — without mutation, diversity decreases monotonically and the EA can stall in a local optimum.
- No adaptive step / variance — there is no mechanism to widen the search if it stagnates.
- The ODE solve dominates the runtime, so larger populations and longer runs become expensive.

**Improvements worth trying**
- Add Gaussian mutation `x' = x + sigma * N(0, I)` with adaptive `sigma` (1/5 success rule, or full self-adaptation as in evolution strategies).
- Replace arithmetic crossover with **BLX-α** or **SBX**, which generate offspring inside / outside the parents' bounding box and preserve diversity.
- Use **CMA-ES** for a more principled adaptive search in continuous spaces.

**EA vs Metropolis-Hastings**
Both maintain a Markov-style state and accept/reject candidates based on a quality measure, but:
- An EA maintains a **population** and generates many candidates per step from recombinations of existing ones, while MH operates on a single chain and one candidate per step.
- The EA targets the **mode** of the fitness landscape (it's an optimizer); MH targets the **stationary distribution** (it's a sampler).
- An EA does not need a probabilistic interpretation of the objective; MH does.